# Planars — Annotation Status

Shows the current span results based on your filled annotation sheets.

**To use:**
1. Set `DRIVE_FOLDER_PATH` in the cell below to your planars Drive folder
2. From the menu: **Runtime → Run all**
3. Google will ask for permission once — click through to allow
4. The chart will appear at the bottom of the page

In [ ]:
# ❗ Set this to your planars Drive folder ID
# Find it in the folder URL: drive.google.com/drive/folders/<THIS_PART>
DRIVE_FOLDER_ID = 'your-folder-id-here'

In [ ]:
# Setup and run — nothing to change here
import subprocess, sys, importlib, importlib.metadata

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--pre',
     'planars', 'gspread', 'google-auth'])
importlib.invalidate_caches()
_ver = importlib.metadata.version('planars')
print(f"planars {_ver} ✓")

from google.colab import auth
auth.authenticate_user()

import json, gspread
from google.auth import default
from googleapiclient.discovery import build
from planars.charts import collect_all_spans_from_sheets, domain_chart

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

results = drive_svc.files().list(
    q=f"'{DRIVE_FOLDER_ID}' in parents and name contains 'manifest_' and trashed=false",
    fields="files(id, name)"
).execute()
if not results['files']:
    raise FileNotFoundError(
        f'No manifest file found in Drive folder: {DRIVE_FOLDER_ID}\n'
        'Please check that DRIVE_FOLDER_ID is correct.'
    )

manifest = {}
for f in results['files']:
    lang_id = f['name'].replace('manifest_', '').replace('.json', '')
    content = drive_svc.files().get_media(fileId=f['id']).execute()
    manifest[lang_id] = json.loads(content)

print(f'Connected. Loaded data for: {", ".join(manifest.keys())}')
print('Computing spans...')

_result = collect_all_spans_from_sheets(gc, manifest)
if len(_result) == 2:
    df, lang_meta = _result
else:
    # planars <0.1.0a3 — wrap old 3-tuple into lang_meta shape
    df, _kp, _p2n = _result
    lang_meta = {list(manifest.keys())[0]: {"keystone_pos": _kp, "pos_to_name": _p2n}}
lang_id, meta = next(iter(lang_meta.items()))
print(f'Done. {len(df)} spans computed for {lang_id}.')

In [ ]:
# Domain chart — hover over segments for details
domain_chart(df, meta["keystone_pos"], meta["pos_to_name"])